<a href="https://colab.research.google.com/github/AboimovVlad/poltava-AR/blob/main/Docs/ML_trning/Embeding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3

# Подключаемся к нашей будущей базе данных
conn = sqlite3.connect("poltava_ar.db")
cursor = conn.cursor()

# Создаем таблицы
cursor.execute(
    """CREATE TABLE IF NOT EXISTS monuments (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    name TEXT NOT NULL,
                    description TEXT NOT NULL,
                    construction_year TEXT NOT NULL,
                    lat REAL,
                    lon REAL,
                    audio_path TEXT NOT NULL,
                    youtube_url TEXT NOT NULL
                )"""
)

print("--- Интерактивное наполнение базы данных памятников ---")
print("Введите 'exit' в поле названия, чтобы закончить ввод.\n")

while True:
  name = input("Введите название памятника: ")
  if name.lower() == "exit":
    break
  description = input("Введите историческую справку: ")
  audio_path = input("Введите путь к аудиофайлу (например, audio/slava.mp3): ")
  construction_year = input("Введите год конструирования")
  lat = input("Введите долготу")
  lon = input("Введите широту")
  youtube_url = input("Введите ссылку на видео про памятник")

  # Записываем монумент в таблицу и сразу получаем его ID
  cursor.execute(
      "INSERT INTO monuments (name, description, construction_year, lat, lon, audio_path, youtube_url) VALUES (?, ?, ?, ?, ?, ?, ?)",
      (name, description, construction_year, lat, lon, audio_path, youtube_url),
  )
  conn.commit()

  monument_id = cursor.lastrowid
  print(
      f"✅ Памятник '{name}' успешно добавлен с ID = {monument_id}!\n"
      "Теперь для него можно запускать скрипт нарезки и генерации эмбеддингов"
      " (с этим ID).\n"
  )

conn.close()

--- Интерактивное наполнение базы данных памятников ---
Введите 'exit' в поле названия, чтобы закончить ввод.

Введите название памятника: Заяць гитарист
Введите историческую справку: Ця вінтажна дерев'яна скульптура «Заєць-гітарист» 1989 року є унікальним зразком ручної різьби по дереву. Майстер ретельно опрацював кожну деталь фігурки за допомогою традиційного столярного приладдя, зберігши живу текстуру натуральної деревини. Завдяки тонкій роботі різцем образ пухнастого музиканта з гітарою виглядає надзвичайно теплим, виразним і динамічним. Цей артефакт кінця 80-х років стане чудовим акцентом у колекції поціновувачів авторських виробів і ретро-декору.
Введите путь к аудиофайлу (например, audio/slava.mp3): audio/Rebibt_Audio.mp3
Введите год конструирования1989
Введите долготу
Введите широту
Введите ссылку на видео про памятникhttps://youtu.be/0-BXkPDkSBE?si=s3GMHAoycf581M4Z
✅ Памятник 'Заяць гитарист' успешно добавлен с ID = 1!
Теперь для него можно запускать скрипт нарезки и генерации

In [ ]:
import os
import shutil
import zipfile
from google.colab import files

extract_folder = "temp_monument_photos"

# Очищаємо папку від попередніх фото, якщо вони там були
if os.path.exists(extract_folder):
  shutil.rmtree(extract_folder)
os.makedirs(extract_folder, exist_ok=True)

print("Оберіть ZIP-архів із фотографіями пам'ятника:")
uploaded = files.upload()

if not uploaded:
  print("⚠️ Файл не обрано.")
else:
  zip_filename = next(iter(uploaded))
  with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

  # Видаляємо сам архів після розпаковки, щоб не засмічувати диск Colab
  os.remove(zip_filename)
  print(
      f"✅ Архів '{zip_filename}' успішно розпаковано у папку"
      f" '{extract_folder}'."
  )


Оберіть ZIP-архів із фотографіями пам'ятника:


Saving p1sun.zip to p1sun.zip
✅ Архів 'p1sun.zip' успішно розпаковано у папку 'temp_monument_photos'.


In [ ]:
import os
import sqlite3
import numpy as np
import timm
import torch
from PIL import Image
from PIL.ExifTags import GPSTAGS, TAGS

extract_folder = "temp_monument_photos"

if not os.path.exists(extract_folder) or not os.listdir(extract_folder):
  raise FileNotFoundError(
      f"❌ Папка '{extract_folder}' порожня. Спочатку виконайте скрипт"
      " розпаковки."
  )

# 1. Підключення до БД та створення таблиці за схемою
conn = sqlite3.connect("poltava_ar.db")
conn.execute("PRAGMA foreign_keys = ON;")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS MonumentEmbeddings (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        monument_id INTEGER NOT NULL,
        embedding BLOB NOT NULL,
        latitude REAL,
        longitude REAL,
        FOREIGN KEY (monument_id) REFERENCES monuments(id) ON DELETE CASCADE
    )
""")
conn.commit()

# 2. Вибір існуючого пам'ятника за ID (Foreign Key)
cursor.execute("SELECT id, name, lat, lon FROM monuments")
monuments = cursor.fetchall()

if not monuments:
  conn.close()
  raise ValueError(
      "❌ У таблиці monuments немає записів. Спочатку додайте пам'ятник."
  )

print("Доступні пам'ятники в базі:")
for m in monuments:
  print(f"  ID: {m[0]} — {m[1]}")

while True:
  try:
    monument_id = int(
        input("\nВведіть ID пам'ятника (monument_id) для прив'язки фото: ")
    )
    cursor.execute(
        "SELECT lat, lon FROM monuments WHERE id = ?", (monument_id,)
    )
    monument_coords = cursor.fetchone()
    if monument_coords:
      default_lat, default_lon = monument_coords
      break
    print("⚠️ Памятник із таким ID не найден. Спробуйте ще раз.")
  except ValueError:
    print("⚠️ Введіть ціле число.")

# 3. Завантаження моделі EfficientNet-Lite0
print("\n⏳ Завантажуємо EfficientNet-Lite0...")
device = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model(
    "tf_efficientnet_lite0.in1k", pretrained=True, num_classes=0
)
model = model.to(device).eval()

data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)


def get_image_embedding(image_path: str) -> bytes:
  """Перетворює зображення на L2-нормалізований вектор float32 (BLOB)."""
  image = Image.open(image_path).convert("RGB")
  input_tensor = transform(image).unsqueeze(0).to(device)

  with torch.no_grad():
    features = model(input_tensor)
    features = torch.nn.functional.normalize(features, p=2, dim=1)

  return features.cpu().numpy().astype(np.float32).flatten().tobytes()


def get_exif_coords(image_path: str):
  """Витягує GPS-координати з EXIF або повертає координати пам'ятника."""
  try:
    image = Image.open(image_path)
    exif_data = image._getexif()
    if not exif_data:
      return default_lat, default_lon

    gps_info = {}
    for tag, value in exif_data.items():
      if TAGS.get(tag) == "GPSInfo":
        for t in value:
          gps_info[GPSTAGS.get(t, t)] = value[t]

    if "GPSLatitude" in gps_info and "GPSLongitude" in gps_info:

      def to_deg(val):
        return float(val[0]) + float(val[1]) / 60.0 + float(val[2]) / 3600.0

      lat = to_deg(gps_info["GPSLatitude"])
      if gps_info.get("GPSLatitudeRef") != "N":
        lat = -lat
      lon = to_deg(gps_info["GPSLongitude"])
      if gps_info.get("GPSLongitudeRef") != "E":
        lon = -lon
      return lat, lon
  except Exception:
    pass
  return default_lat, default_lon


# 4. Векторизація кожного фото та запис у БД (1 фото = 1 запис)
valid_extensions = (".jpg", ".jpeg", ".png", ".webp")
added_count = 0

for root, _, files_list in os.walk(extract_folder):
  for file_name in sorted(files_list):
    if file_name.startswith(".") or "__MACOSX" in root:
      continue
    if file_name.lower().endswith(valid_extensions):
      full_path = os.path.join(root, file_name)

      emb_blob = get_image_embedding(full_path)
      lat, lon = get_exif_coords(full_path)

      cursor.execute(
          """INSERT INTO MonumentEmbeddings (monument_id, embedding, latitude, longitude)
                   VALUES (?, ?, ?, ?)""",
          (monument_id, emb_blob, lat, lon),
      )
      added_count += 1
      print(f"📸 {file_name} -> додано у БД (monument_id = {monument_id})")

conn.commit()
conn.close()

print(f"\n🎉 Готово! Додано записів у MonumentEmbeddings: {added_count}.")


Доступні пам'ятники в базі:
  ID: 1 — Заяць гитарист

Введіть ID пам'ятника (monument_id) для прив'язки фото: 1

⏳ Завантажуємо EfficientNet-Lite0...
📸 photo_10_2026-11-14_08-47-20.jpg -> додано у БД (monument_id = 1)
📸 photo_10_2026-11-14_11-35-03.jpg -> додано у БД (monument_id = 1)
📸 photo_11_2026-11-14_08-47-20.jpg -> додано у БД (monument_id = 1)
📸 photo_11_2026-11-14_11-35-03.jpg -> додано у БД (monument_id = 1)
📸 photo_12_2026-11-14_08-47-20.jpg -> додано у БД (monument_id = 1)
📸 photo_12_2026-11-14_11-35-03.jpg -> додано у БД (monument_id = 1)
📸 photo_13_2026-11-14_08-47-20.jpg -> додано у БД (monument_id = 1)
📸 photo_13_2026-11-14_11-35-03.jpg -> додано у БД (monument_id = 1)
📸 photo_14_2026-11-14_08-47-20.jpg -> додано у БД (monument_id = 1)
📸 photo_14_2026-11-14_11-35-03.jpg -> додано у БД (monument_id = 1)
📸 photo_15_2026-11-14_08-47-20.jpg -> додано у БД (monument_id = 1)
📸 photo_16_2026-11-14_08-47-20.jpg -> додано у БД (monument_id = 1)
📸 photo_17_2026-11-14_08-47-20.jpg

In [ ]:
import os
import sqlite3
import numpy as np
import timm
import torch
from PIL import Image
from google.colab import files

# 1. Завантаження тестової фотографії
print("Оберіть тестове фото пам'ятника (імітація камери смартфона):")
uploaded = files.upload()

if not uploaded:
  raise ValueError("⚠️ Фото не обрано.")

test_image_path = next(iter(uploaded))

# 2. Завантаження тієї самої моделі EfficientNet-Lite0
print("\n⏳ Завантажуємо EfficientNet-Lite0...")
device = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model(
    "tf_efficientnet_lite0.in1k", pretrained=True, num_classes=0
)
model = model.to(device).eval()

data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)


def get_query_vector(image_path: str) -> np.ndarray:
  """Отримує L2-нормалізований вектор (1280 float32) для тестового фото."""
  image = Image.open(image_path).convert("RGB")
  input_tensor = transform(image).unsqueeze(0).to(device)

  with torch.no_grad():
    features = model(input_tensor)
    features = torch.nn.functional.normalize(features, p=2, dim=1)

  return features.cpu().numpy().astype(np.float32).flatten()


query_vector = get_query_vector(test_image_path)

# 3. Отримання всіх збережених векторів із бази даних
conn = sqlite3.connect("poltava_ar.db")
cursor = conn.cursor()

cursor.execute("""
    SELECT
        me.id,
        me.monument_id,
        m.name,
        me.embedding
    FROM MonumentEmbeddings me
    JOIN monuments m ON me.monument_id = m.id
""")
rows = cursor.fetchall()
conn.close()

if not rows:
  raise ValueError("❌ У таблиці MonumentEmbeddings поки немає жодного вектора.")

# 4. Розрахунок Cosine Similarity (скалярний добуток нормалізованих векторів)
results = []

for emb_id, monument_id, monument_name, emb_blob in rows:
  # Розпаковуємо BLOB назад у масив із 1280 чисел float32
  db_vector = np.frombuffer(emb_blob, dtype=np.float32)

  # Оскільки обидва вектори вже L2-нормалізовані, скалярний добуток (np.dot) == Cosine Similarity
  similarity = float(np.dot(query_vector, db_vector))

  results.append({
      "emb_id": emb_id,
      "monument_id": monument_id,
      "name": monument_name,
      "score": similarity,
  })

# Сортуємо від найбільш схожого до найменш схожого
results.sort(key=lambda x: x["score"], reverse=True)

# 5. Вивід результатів розпізнавання
print("\n" + "=" * 55)
print(f"🔍 РЕЗУЛЬТАТИ РОЗПІЗНАВАННЯ ДЛЯ: {test_image_path}")
print("=" * 55)

best_match = results[0]
print(
    f"🏆 Найкращий збіг: {best_match['name']} (ID: {best_match['monument_id']})"
)
print(
    f"📊 Схожість (Cosine Similarity): {best_match['score']:.4f}"
    f" ({best_match['score']*100:.1f}%)"
)

# Орієнтовний поріг впевненості (можна підбирати на реальних тестах)
THRESHOLD = 0.70
if best_match["score"] >= THRESHOLD:
  print("✅ СТАТУС: Пам'ятник розпізнано! Можна запускати AR та аудіогід.")
else:
  print(
      "⚠️ СТАТУС: Низька впевненість (нижче порогу 0.70). Попросіть користувача"
      " підійти ближче або змінити ракурс."
  )

print("\nТоп-5 найближчих ракурсів у базі:")
for i, item in enumerate(results[:5], start=1):
  print(
      f"  {i}. [{item['score']:.4f}] — {item['name']} (monument_id:"
      f" {item['monument_id']}, запис #{item['emb_id']})"
  )

# Видаляємо тимчасовий тестовий файл
os.remove(test_image_path)


Оберіть тестове фото пам'ятника (імітація камери смартфона):


Saving тест фото.jpg to тест фото.jpg

⏳ Завантажуємо EfficientNet-Lite0...

🔍 РЕЗУЛЬТАТИ РОЗПІЗНАВАННЯ ДЛЯ: тест фото.jpg
🏆 Найкращий збіг: Заяць гитарист (ID: 1)
📊 Схожість (Cosine Similarity): 0.7420 (74.2%)
✅ СТАТУС: Пам'ятник розпізнано! Можна запускати AR та аудіогід.

Топ-5 найближчих ракурсів у базі:
  1. [0.7420] — Заяць гитарист (monument_id: 1, запис #4)
  2. [0.7225] — Заяць гитарист (monument_id: 1, запис #14)
  3. [0.6949] — Заяць гитарист (monument_id: 1, запис #3)
  4. [0.6932] — Заяць гитарист (monument_id: 1, запис #15)
  5. [0.6819] — Заяць гитарист (monument_id: 1, запис #18)


In [ ]:
import os
import subprocess
import sys
import timm
import torch
import torch.nn as nn
from google.colab import files

try:
  import litert_torch
except ImportError:
  subprocess.check_call(
      [sys.executable, "-m", "pip", "install", "-q", "litert-torch"]
  )
  import litert_torch


# Обгортка, яка робить модель зручною для Android (NHWC + нормалізація всередині)
class AndroidReadyEmbedder(nn.Module):

  def __init__(self, base_model):
    super().__init__()
    self.base_model = base_model

  def forward(self, x):
    # x приходить з Android у форматі [1, 224, 224, 3] зі значеннями [0.0 .. 255.0]
    # 1. Переставляємо осі з NHWC [1, 224, 224, 3] у NCHW [1, 3, 224, 224]
    x = x.permute(0, 3, 1, 2)
    # 2. Нормалізуємо пікселі в [-1.0, 1.0] (як вимагає EfficientNet-Lite0)
    x = (x - 127.5) / 127.5
    # 3. Отримуємо вектор ознак [1, 1280]
    features = self.base_model(x)
    # 4. Одразу робимо L2-нормалізацію для Cosine Similarity
    return torch.nn.functional.normalize(features, p=2, dim=1)


base_model = (
    timm.create_model(
        "tf_efficientnet_lite0.in1k", pretrained=True, num_classes=0
    )
    .cpu()
    .eval()
)
wrapped_model = AndroidReadyEmbedder(base_model).eval()

# Тепер вхідний тензор має стандартну для Android форму [1, 224, 224, 3]
with torch.no_grad():
  sample_input = (torch.rand(1, 224, 224, 3) * 255.0,)

tflite_filename = "efficientnet_lite0_android.tflite"
edge_model = litert_torch.convert(wrapped_model, sample_input)
edge_model.export(tflite_filename)
print(f"✅ Готово! Модель для Android збережена як '{tflite_filename}'")

files.download(tflite_filename)
if os.path.exists("poltava_ar.db"):
  files.download("poltava_ar.db")


W1114 07:51:50.175000 930 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:03) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

(00:05) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:02)

(00:05) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default (+00:05)

... (20 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>